In [22]:
import pathlib as Path
import numpy as np
import pandas as pd


## Create dataframe from embeddings

In [69]:
#-----Frequency Embeddings dataframe creation-----
def create_embeddings_dataframe(root_path):
    emb_root = Path.Path(root_path)
    emb_files = emb_root.rglob("embeddings_samples.npz")
    
    all_data = []
    
    for emb_file in emb_files:
        emb = np.load(emb_file)
        X = emb['X']
        y = emb['y']
        subjects = emb['subs']
        
        for i in range(X.shape[0]):
            data_point = {
                'embedding': X[i],
                'label': y[i],
                'subject': subjects[i],
                'file_path': str(emb_file)
            }
            all_data.append(data_point)
    
    df = pd.DataFrame(all_data)
    return df


## Clean embedding DF

In [70]:

def df_cleaning(df, emb_size=384):
        # --- Expand embedding column into 381 separate columns ---
    embedding_df = pd.DataFrame(df["embedding"].tolist(),
                                columns=[f"emb_{i}" for i in range(emb_size)])

    # --- Concatenate back to the original DataFrame (optional) ---
    df_expanded = pd.concat([df.drop(columns=["embedding"]), embedding_df], axis=1)
    return df_expanded

In [71]:
#-----Frequency Embeddings dataframe creation-----
freq_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//frequency_emb_stored//")
#-----Temporal Embeddings dataframe creation-----
temp_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//temporal_emb_stored//")
#-----Combined Embeddings dataframe creation-----
comb_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//emb_stored//")

freq_emb_df= df_cleaning(freq_emb, emb_size=384)
temp_emb_df= df_cleaning(temp_emb, emb_size=384)
comb_emb_df= df_cleaning(comb_emb, emb_size=768)

In [77]:
comb_emb_df = comb_emb_df.loc[~comb_emb_df.subject.duplicated(keep='first'), :]
temp_emb_df = temp_emb_df.loc[~temp_emb_df.subject.duplicated(keep='first'), :]
freq_emb_df = freq_emb_df.loc[~freq_emb_df.subject.duplicated(keep='first'), :]

# Metadata labeling
    - 0 -> HC
    - 1 -> unknown
    - 2 -> MCI-AD
    - 3 -> MCI- LBD

In [78]:
metadata_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//preDLB_shared(PSY_RAW).csv')
df_metadata = pd.read_csv(metadata_path, sep=";", encoding="utf-8-sig")

In [79]:
clinical_data_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//clinical_data_csv.csv')
df_clinical = pd.read_csv(clinical_data_path, sep=",", encoding="utf-8-sig")

In [80]:
handcrafted_features_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_hf//corpus_LBD_CZ_002_writing_results_table_original_filtered_extended.csv')
df_handcrafted_features = pd.read_csv(handcrafted_features_path, sep=";", encoding="utf-8-sig")

In [51]:
df_metadata

,ID_1.meranie,oficiálna dg,HC0_nHC1_MCI2_MCILB3_baseline,Vzdelani,Delka_vzdelani,JLO_HS,JLO_perc,Unnamed: 7,[1] JLO_Z,Vizuospacialni_funkce_Z,...,Unnamed: 149,CRT_Z.1,Premorbidni_Inteligence_Z.2,Verf_Lex_HS.2,Verf_Lex_Z.1,Verf_sem_HS.2,Verf_sem_Z.1,Razeni_obrazku_HS.2,Razeni_obrazku_Z.1,Exekutivni_funkce_Z.1
0,pre-LBD-1,NaN,1.0,3.0,13,26.0,56,"0,56","0,150969215","0,150969215",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,pre-LBD-2,NaN,3.0,3.0,13,27.0,72,"0,72","0,582841507","0,582841507",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pre-LBD-3,NaN,1.0,3.0,13,26.0,56,"0,56","0,150969215","0,150969215",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,pre-LBD-4,NaN,1.0,2.0,12,23.0,40,"0,4","-0,253347103","-0,253347103",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,pre-LBD-5,NaN,3.0,3.0,17,23.0,40,"0,4","-0,253347103","-0,253347103",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,pre-LBD-122,NaN,3.0,3.0,13,21.0,NaN,NaN,"-0,77",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
156,pre-LBD-124 (Harth),NaN,1.0,4.0,20,29.0,NaN,NaN,"1,08","0,54",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157,pre-LBD-125,NaN,2.0,3.0,13,23.0,40,NaN,"-0,25",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158,pre-LBD-126,NaN,2.0,3.0,13,30.0,86,"1,08","1,08032",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
df_clinical

,filename,#id,#personalID,MR ID,group,age,gender,LED,MOCA,education type,education length,memory z-score,visuo-spatial z-score,attention z-score,executive function z-score,GDS,pozn.
0,COBEN_COBEN_ACOUSTIC_VZ_AD01.wav,COBEN_CZCOBEN033,CZCOBEN033,1819A,aMCI,62.0,0.0,NaN,27.0,3.0,14,"0,22","1,08","-0,94","-1,95",6.0,roky education doplníme prumerem
1,COBEN_COBEN_ACOUSTIC_IK_AD02.wav,COBEN_CZCOBEN034,CZCOBEN034,1939A,aMCI,66.0,0.0,NaN,20.0,2.0,12,"-1,63","-0,77","-0,78","-1,95",0.0,NaN
2,COBEN_COBEN_ACOUSTIC_OS_AD03.wav,COBEN_CZCOBEN035,CZCOBEN035,2367A,aMCI,74.0,1.0,NaN,7.0,2.0,11,-3,"-1,34","-2,15","-1,51",4.0,NaN
3,COBEN_COBEN_ACOUSTIC_DV_AD04.wav,COBEN_CZCOBEN036,CZCOBEN036,2365A,aMCI,75.0,1.0,NaN,25.0,2.0,10,"-1,36","0,15","-0,38","-0,76",10.0,NaN
4,COBEN_COBEN_ACOUSTIC_HS_AD06.wav,COBEN_CZCOBEN038,CZCOBEN038,2305A,aMCI,69.0,1.0,NaN,22.0,2.0,12,"-0,15","-1,75","-1,03","-1,48",24.0,roky education doplníme prumerem
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,preDLB_pre-LBD-112.wav,preDLB_pre-LBD-112#1,pre-LBD-112#1,6009A,MCI-LB,84.0,1.0,NaN,24.0,3.0,13,"-0,69",NaN,-2,-1,7.0,NaN
220,preDLB_pre-LBD-114.wav,preDLB_pre-LBD-114#1,pre-LBD-114#1,6216A,MCI-LB,70.0,1.0,NaN,24.0,3.0,13,0,"-1,34","0,17","-0,56",8.0,NaN
221,preDLB_pre-LBD-120.wav,preDLB_pre-LBD-120#1,pre-LBD-120#1,6457A,MCI-LB,71.0,0.0,NaN,29.0,2.0,12,"-0,22","-0,25",0,"-1,61",7.0,NaN
222,preDLB_pre-LBD-115.wav,preDLB_pre-LBD-115#1,pre-LBD-115#1,5998A,PD,65.0,1.0,300,24.0,2.0,11,"-0,67","-2,17","-1,33","-1,17",11.0,NaN


In [81]:
import re
import pandas as pd

def append_col_when_main_contains_source(
    df_main, df_source, *, 
    match_col_main="subject",            # in df_main
    match_col_source="ID_1.meranie",     # in df_source
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + keep only rows in source with non-missing values
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()
    df_source = df_source.dropna(subset=[value_col])

    # init target column
    if new_col_name not in df_main:
        df_main[new_col_name] = pd.NA

    # for each source row, mark all main rows whose subject CONTAINS the source token
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token:
            continue
        mask = df_main[match_col_main].str.contains(re.escape(token), na=False, case=not case)
        # write only where we don't have a value yet (keeps first hit)
        to_set = mask & df_main[new_col_name].isna()
        df_main.loc[to_set, new_col_name] = r[value_col]

    return df_main

In [82]:
freq_emb_df_lbl = append_col_when_main_contains_source(
    df_main=freq_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [ ]:
freq_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/freq_emb_df_half_lbl.csv", sep=";")

In [85]:

temp_emb_df_lbl = append_col_when_main_contains_source(
    df_main=temp_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [ ]:

temp_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/temp_emb_df_half__lbl.csv", sep=";")

In [87]:

comb_emb_df_lbl = append_col_when_main_contains_source(
    df_main=comb_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [ ]:
comb_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/comb_emb_df_half__lbl.csv", sep=";")

In [89]:

df_handcrafted_lbl = append_col_when_main_contains_source(
    df_main=df_handcrafted_features,
    df_source=df_metadata,
    match_col_main="ID",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)


In [ ]:

df_handcrafted_lbl.to_csv("LBD_CZ_002_COBEN_dfs/df_handcrafted_half_lbl.csv", sep=";")

In [145]:
temp_emb_df_complet_lbl = pd.read_csv("LBD_CZ_002_COBEN_dfs/temp_emb_df_lbl.csv", sep=";")

freq_emb_df_complet_lbl= pd.read_csv("./LBD_CZ_002_COBEN_dfs/freq_emb_df_lbl.csv", sep=";")

comb_emb_df_complet_lbl= pd.read_csv("LBD_CZ_002_COBEN_dfs/comb_emb_df_lbl.csv", sep=";")

hf_emb_df_complet_lbl= pd.read_csv("LBD_CZ_002_COBEN_dfs/df_handcrafted_lbl.csv", sep=";")


In [121]:
def append_multiple_cols_when_main_contains_source(
    df_main, df_source, *,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_cols=("HC0_nHC1_MCI2_MCILB3_baseline",),
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + clean
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()

    # initialize missing columns
    for col in value_cols:
        if col not in df_main.columns:
            df_main[col] = pd.NA

    # iterate through source
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token or pd.isna(token):
            continue

        # use 'case' argument as passed (was inverted before)
        mask = df_main[match_col_main].str.contains(re.escape(str(token)), na=False, case=case)
        to_set = mask

        # assign all defined value columns
        for col in value_cols:
            if col not in r or pd.isna(r[col]):
                continue
            df_main.loc[to_set & df_main[col].isna(), col] = r[col]

    return df_main


In [153]:
clinical_hf_emb_df_complet_lbl = append_multiple_cols_when_main_contains_source(
    df_main=hf_emb_df_complet_lbl,
    df_source=df_clinical,
    match_col_main="ID",
    match_col_source="#personalID",
    value_cols=["age",
                "gender",
                "MOCA",
                "education type",
                "education length",
                "memory z-score",
                "visuo-spatial z-score",
                "attention z-score",
                "executive function z-score",
                "GDS"]
)

In [ ]:
temp_emb_df_complet_lbl_2 = temp_emb_df_complet_lbl.drop(temp_emb_df_complet_lbl.columns[0], axis=1)
freq_emb_df_complet_lbl_2 = freq_emb_df_complet_lbl.drop(freq_emb_df_complet_lbl.columns[0], axis=1)
comb_emb_df_complet_lbl_2 = comb_emb_df_complet_lbl.drop(comb_emb_df_complet_lbl.columns[0], axis=1)
hf_emb_df_complet_lbl_2 = hf_emb_df_complet_lbl.drop(hf_emb_df_complet_lbl.columns[0], axis=1)

In [148]:
temp_emb_df_complet_lbl

,label,subject,file_path,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,...,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383,diagnosis
0,1,COBEN-WTABLET-AS-HCD05,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.093985,-0.018822,0.014201,0.012169,0.014989,-0.060903,0.014550,...,-0.019736,-0.120115,0.044475,-0.054144,-0.038272,0.016141,0.047260,-0.016306,0.001134,0.0
1,1,COBEN-WTABLET-BB-PD08,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.081599,-0.013635,0.011962,0.018857,0.009594,-0.043635,0.006841,...,-0.005933,-0.128685,0.050552,-0.062111,-0.041462,0.005450,0.039324,-0.029658,-0.016626,4.0
2,1,COBEN-WTABLET-DM-AD15,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.077031,-0.017034,0.011105,0.013916,0.006772,-0.038075,0.009316,...,-0.006596,-0.128357,0.049246,-0.063082,-0.040102,0.008801,0.045370,-0.026662,-0.017330,2.0
3,1,COBEN-WTABLET-DM-PD17,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.071908,-0.007060,0.017014,0.019616,0.010703,-0.032701,-0.001381,...,-0.006994,-0.125160,0.050763,-0.065362,-0.037256,0.003106,0.035329,-0.037928,-0.022555,4.0
4,1,COBEN-WTABLET-DS-HC36,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.083261,-0.022561,0.014251,0.011554,0.009685,-0.055109,0.020986,...,-0.016858,-0.120602,0.043949,-0.054705,-0.037846,0.013065,0.043531,-0.013824,0.002849,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,1,pre-LBD-95#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.083182,-0.021007,0.014818,0.015376,0.014974,-0.059431,0.016709,...,-0.016610,-0.120874,0.042836,-0.054272,-0.047351,0.012994,0.045947,-0.012357,0.003878,1.0
262,1,pre-LBD-96#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.079720,-0.015989,0.018476,0.018010,0.012075,-0.059524,0.016757,...,-0.011069,-0.131888,0.045626,-0.060346,-0.046665,0.003306,0.044285,-0.015272,-0.001581,1.0
263,1,pre-LBD-97#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.077280,-0.014559,0.012996,0.012618,0.010235,-0.043303,0.002792,...,-0.014923,-0.116215,0.047198,-0.058298,-0.036434,0.019876,0.041553,-0.019976,-0.015242,1.0
264,1,pre-LBD-98#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.089632,-0.019975,0.011347,0.014207,0.012226,-0.061016,0.016647,...,-0.021903,-0.115395,0.045972,-0.050632,-0.036941,0.016790,0.045878,-0.012824,0.001774,1.0


In [154]:
clinical_hf_emb_df_complet_lbl

,ID,w.cz.fnusa.15_1_95th percentile of acceleration (in-air),w.cz.fnusa.16_1_95th percentile of acceleration (in-air),w.cz.fnusa.17_1_95th percentile of acceleration (in-air),w.cz.fnusa.18_1_95th percentile of acceleration (in-air),w.cz.fnusa.19_1_95th percentile of acceleration (in-air),w.cz.fnusa.9_1_95th percentile of acceleration (in-air),w.cz.fnusa.15_1_95th percentile of acceleration (on-surface),w.cz.fnusa.16_1_95th percentile of acceleration (on-surface),w.cz.fnusa.17_1_95th percentile of acceleration (on-surface),...,age,gender,MOCA,education type,education length,memory z-score,visuo-spatial z-score,attention z-score,executive function z-score,GDS
0,COBEN-WTABLET-AS-HCD05,"1382,947668","2062,000865","2892,979665","2192,046228","2013,070361","2235,580284","586,7346939","1627,424458","1711,843728",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,COBEN-WTABLET-BB-PD08,"2184,038058","2960,491263","2833,151798",NaN,"2643,941418","2704,362771","1705,984308","1274,972304","1643,871705",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,COBEN-WTABLET-DM-AD15,"2129,216623","3373,868306","1686,794432",NaN,"3212,659823","3435,479391","1657,058812","1573,868658","2365,68499",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,COBEN-WTABLET-DM-PD17,"3973,130385","3650,787752","2810,386562",NaN,"3278,226429","2390,344711","2226,888919","2314,558206","1991,804497",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,COBEN-WTABLET-DS-HC36,"973,0839762","1194,677753","2163,52771",NaN,"1601,144846","1533,402248","581,0757432","1754,342814","1998,926041",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,pre-LBD-95#1,"1771,638967","2223,266011","2726,863796",NaN,NaN,NaN,"1248,437707","2379,581423","2340,413141",...,55.0,1.0,27.0,2.0,12,"-0,43","-0,25",0,"0,22",13.0
262,pre-LBD-96#1,"1587,495217","3116,456725","5244,55234",NaN,"3146,502315","3071,576595","1670,497532","5487,15554","6979,041639",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
263,pre-LBD-97#1,"1529,780961","2846,530626","3170,3414",NaN,"2702,591544","1901,945515","914,3545637","4092,897902","2793,424795",...,70.0,0.0,26.0,3.0,13,"-0,66","1,08","-0,33","-1,17",4.0
264,pre-LBD-98#1,"1875,912627","2772,534223","3176,331668",NaN,"2088,759949","2417,683099","775,4663476","1565,536158","1896,258373",...,75.0,1.0,14.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,19.0


# Spearsman correlation / finding covariants

In [155]:

cols_to_convert = clinical_hf_emb_df_complet_lbl.columns[1:]

clinical_hf_emb_df_complet_lbl[cols_to_convert] = (
    clinical_hf_emb_df_complet_lbl[cols_to_convert]
    .apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce'))
)

In [156]:
clinical_hf_emb_df_complet_lbl.dtypes

ID                                                           object
w.cz.fnusa.15_1_95th percentile of acceleration (in-air)    float64
w.cz.fnusa.16_1_95th percentile of acceleration (in-air)    float64
w.cz.fnusa.17_1_95th percentile of acceleration (in-air)    float64
w.cz.fnusa.18_1_95th percentile of acceleration (in-air)    float64
                                                             ...   
memory z-score                                              float64
visuo-spatial z-score                                       float64
attention z-score                                           float64
executive function z-score                                  float64
GDS                                                         float64
Length: 462, dtype: object

In [168]:

from scipy import stats
import numpy as np

meta_cols=["age",
         #   "gender",
            "MOCA",
            "education type",
            "education length",
            "memory z-score",
            "visuo-spatial z-score",
            "attention z-score",
            "executive function z-score",
            "GDS"]

feature_cols = clinical_hf_emb_df_complet_lbl.drop(columns=['ID', 'diagnosis','gender'] + meta_cols).columns.tolist()


def correlation_to_df(df, feature_cols, meta_cols, corr_type='spearman'):
    from scipy import stats
    corr_results = []
    for feature in feature_cols:
        for meta in meta_cols:
            feature_data = pd.to_numeric(df[feature], errors='coerce')
            meta_data = pd.to_numeric(df[meta], errors='coerce')
            if corr_type == 'spearman':
                corr, p_value = stats.spearmanr(feature_data, meta_data, nan_policy='omit')
            elif corr_type == 'pearson':
                corr, p_value = stats.pearsonr(feature_data.dropna(), meta_data.dropna())
            else:
                raise ValueError("corr_type must be 'spearman' or 'pearson'")
            corr_results.append({
                'feature': feature,
                'meta_variable': meta,
                'correlation': corr,
                'p_value': p_value
            })
    corr_df = pd.DataFrame(corr_results)
    return corr_df


In [170]:
import numpy as np
import pandas as pd
from scipy import stats

def safe_spearman(x, y, min_n=3):
    """
    x, y: 1D array-like
    returns (rho, p, n_used)
    """
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    y = pd.to_numeric(pd.Series(y), errors="coerce")

    mask = x.notna() & y.notna()
    x2 = x[mask].values
    y2 = y[mask].values
    n = len(x2)

    if n < min_n:
        return np.nan, np.nan, n

    # constant vectors -> undefined correlation
    if np.all(x2 == x2[0]) or np.all(y2 == y2[0]):
        return np.nan, np.nan, n

    rho, p = stats.spearmanr(x2, y2)
    # spearmanr can still return nan if ties/degenerate
    if not np.isfinite(rho):
        rho, p = np.nan, np.nan
    return rho, p, n


def correlation_to_df(df, feature_cols, meta_cols, corr_type="spearman", min_n=3):
    rows = []
    for feat in feature_cols:
        feature_data = df[feat]

        for meta in meta_cols:
            meta_data = df[meta]

            if corr_type == "spearman":
                corr, p_value, n_used = safe_spearman(feature_data, meta_data, min_n=min_n)
            elif corr_type == "pearson":
                # paired dropna for pearson too
                x = pd.to_numeric(feature_data, errors="coerce")
                y = pd.to_numeric(meta_data, errors="coerce")
                mask = x.notna() & y.notna()
                x2, y2 = x[mask].values, y[mask].values
                n_used = len(x2)
                if n_used < min_n or np.all(x2 == x2[0]) or np.all(y2 == y2[0]):
                    corr, p_value = np.nan, np.nan
                else:
                    corr, p_value = stats.pearsonr(x2, y2)
            else:
                raise ValueError("corr_type must be 'spearman' or 'pearson'")

            rows.append({
                "feature": feat,
                "meta": meta,
                "corr": corr,
                "p_value": p_value,
                "n_used": n_used
            })

    return pd.DataFrame(rows)


In [162]:
clinical_hf_emb_df_complet_lbl

,ID,w.cz.fnusa.15_1_95th percentile of acceleration (in-air),w.cz.fnusa.16_1_95th percentile of acceleration (in-air),w.cz.fnusa.17_1_95th percentile of acceleration (in-air),w.cz.fnusa.18_1_95th percentile of acceleration (in-air),w.cz.fnusa.19_1_95th percentile of acceleration (in-air),w.cz.fnusa.9_1_95th percentile of acceleration (in-air),w.cz.fnusa.15_1_95th percentile of acceleration (on-surface),w.cz.fnusa.16_1_95th percentile of acceleration (on-surface),w.cz.fnusa.17_1_95th percentile of acceleration (on-surface),...,age,gender,MOCA,education type,education length,memory z-score,visuo-spatial z-score,attention z-score,executive function z-score,GDS
0,COBEN-WTABLET-AS-HCD05,1382.947668,2062.000865,2892.979665,2192.046228,2013.070361,2235.580284,586.734694,1627.424458,1711.843728,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,COBEN-WTABLET-BB-PD08,2184.038058,2960.491263,2833.151798,NaN,2643.941418,2704.362771,1705.984308,1274.972304,1643.871705,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,COBEN-WTABLET-DM-AD15,2129.216623,3373.868306,1686.794432,NaN,3212.659823,3435.479391,1657.058812,1573.868658,2365.684990,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,COBEN-WTABLET-DM-PD17,3973.130385,3650.787752,2810.386562,NaN,3278.226429,2390.344711,2226.888919,2314.558206,1991.804497,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,COBEN-WTABLET-DS-HC36,973.083976,1194.677753,2163.527710,NaN,1601.144846,1533.402248,581.075743,1754.342814,1998.926041,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,pre-LBD-95#1,1771.638967,2223.266011,2726.863796,NaN,NaN,NaN,1248.437707,2379.581423,2340.413141,...,55.0,1.0,27.0,2.0,12.0,-0.43,-0.25,0.00,0.22,13.0
262,pre-LBD-96#1,1587.495217,3116.456725,5244.552340,NaN,3146.502315,3071.576595,1670.497532,5487.155540,6979.041639,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263,pre-LBD-97#1,1529.780961,2846.530626,3170.341400,NaN,2702.591544,1901.945515,914.354564,4092.897902,2793.424795,...,70.0,0.0,26.0,3.0,13.0,-0.66,1.08,-0.33,-1.17,4.0
264,pre-LBD-98#1,1875.912627,2772.534223,3176.331668,NaN,2088.759949,2417.683099,775.466348,1565.536158,1896.258373,...,75.0,1.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN,19.0


In [171]:
df_hf_clinical_corr = correlation_to_df(clinical_hf_emb_df_complet_lbl, feature_cols, meta_cols, corr_type='spearman')

In [176]:
df_hf_clinical_corr

,feature,meta,corr,p_value,n_used
0,w.cz.fnusa.15_1_95th percentile of acceleratio...,age,0.044666,0.660662,99
1,w.cz.fnusa.15_1_95th percentile of acceleratio...,MOCA,0.043330,0.673447,97
2,w.cz.fnusa.15_1_95th percentile of acceleratio...,education type,0.011873,0.908594,96
3,w.cz.fnusa.15_1_95th percentile of acceleratio...,education length,0.078538,0.446892,96
4,w.cz.fnusa.15_1_95th percentile of acceleratio...,memory z-score,0.005033,0.961389,95
...,...,...,...,...,...
4045,w.cz.fnusa.1_1_zero-crossing rate of spiral,memory z-score,0.142506,0.168318,95
4046,w.cz.fnusa.1_1_zero-crossing rate of spiral,visuo-spatial z-score,0.122164,0.240807,94
4047,w.cz.fnusa.1_1_zero-crossing rate of spiral,attention z-score,0.209750,0.041343,95
4048,w.cz.fnusa.1_1_zero-crossing rate of spiral,executive function z-score,0.268764,0.008450,95


In [179]:
df_hf_clinical_corr = df_hf_clinical_corr.drop(df_hf_clinical_corr.columns[-1], axis=1) 

In [183]:
df_hf_clinical_corr

,feature,meta,corr,p_value
0,w.cz.fnusa.15_1_95th percentile of acceleratio...,age,0.044666,0.660662
1,w.cz.fnusa.15_1_95th percentile of acceleratio...,MOCA,0.043330,0.673447
2,w.cz.fnusa.15_1_95th percentile of acceleratio...,education type,0.011873,0.908594
3,w.cz.fnusa.15_1_95th percentile of acceleratio...,education length,0.078538,0.446892
4,w.cz.fnusa.15_1_95th percentile of acceleratio...,memory z-score,0.005033,0.961389
...,...,...,...,...
4045,w.cz.fnusa.1_1_zero-crossing rate of spiral,memory z-score,0.142506,0.168318
4046,w.cz.fnusa.1_1_zero-crossing rate of spiral,visuo-spatial z-score,0.122164,0.240807
4047,w.cz.fnusa.1_1_zero-crossing rate of spiral,attention z-score,0.209750,0.041343
4048,w.cz.fnusa.1_1_zero-crossing rate of spiral,executive function z-score,0.268764,0.008450


In [187]:
# Create separate pivots for correlation and p-value
#df_corr_pivot = df_hf_clinical_corr.pivot(
#    index='feature',
#    columns='meta',
#    values='corr'
#)

df_pval_pivot = df_hf_clinical_corr.pivot(
    index='feature',
    columns='meta',
    values='p_value'
)

In [188]:
df_pval_pivot[df_pval_pivot < 0.05].count()

meta
GDS                           69
MOCA                          65
age                           42
attention z-score             60
education length              52
education type                52
executive function z-score    24
memory z-score                11
visuo-spatial z-score         68
dtype: int64